# holewise_test_Sequential（順次補完）

In [ ]:
# Cell 1 变量设置

import os, time
from pathlib import Path

# ====== 工作目录 ======
WORK_DIR = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2"
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

# ====== 模型 & 数据 ======
RUN_CONFIG_PATH = f"{WORK_DIR}/run_config.json"
SKETCH_JSONL = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2/data/sketch.raw.test.jsonl"

# 训练输出目录（tokenizer/chat_template 在这里）
OUT_DIR_ROOT = f"{WORK_DIR}/runs/stageB_stepwise_qlora"

# adapter 在 checkpoint 里
ADAPTER_DIR  = f"{OUT_DIR_ROOT}"
# ADAPTER_DIR = f"{OUT_DIR_ROOT}/checkpoint-200" #（需要制定checkpoint时）

TOKENIZER_DIR = OUT_DIR_ROOT

# ====== python / torchrun ======
PYTHON = "local-environment/conda_envs/deepseekv2/bin/python"  # 你环境的 python

# ====== 生成参数 ======
MAX_NEW_TOKENS_PER_HOLE = 512
MAX_HOLES_PER_PROOF = 320
DO_SAMPLE = 0
TEMPERATURE = 0.7
TOP_P = 0.95

# ====== 输出目录（每次 test 单独一个文件夹）=====
STAMP = time.strftime("%Y%m%d-%H%M%S")
TEST_RUN_DIR = f"{WORK_DIR}/test_runs/Sequential_filling_test/{STAMP}"
Path(TEST_RUN_DIR).mkdir(parents=True, exist_ok=True)

# 生成结果 shards 会写进这个目录
GEN_DIR = f"{TEST_RUN_DIR}/gen"
Path(GEN_DIR).mkdir(parents=True, exist_ok=True)



# 合并后的 “单洞” jsonl（洞级输出）
MERGED_HOLES_JSONL = f"{TEST_RUN_DIR}/holes_all.jsonl"

# 组装后的 “整题” jsonl（完整 Lean，可直接 verify）
MERGED_GEN_JSONL = f"{TEST_RUN_DIR}/generated_all.jsonl"   

SKIPPED_JSONL = f"{TEST_RUN_DIR}/skipped_groups.jsonl"




# ====== Lean 验证 ======
# 这里你要填你用来判题的 lean 工程目录（lean_judge0/1/2 任选一个）
LEAN_PROJ = "original-layout/Projects/Jupyter/deepseek-prover-V2/LEAN_JUDGE/lean_judge0"

# 验证输出
VERIFY_JSONL = f"{TEST_RUN_DIR}/verify_results.jsonl"
VERIFY_TIMEOUT = 1200  # 每题 Lean 最大秒数
VERIFY_JOBS = 8       # 并发数（别太大）


LAKE = "original-layout/.elan/bin/lake"
ELAN = "original-layout/.elan/bin/elan"
ELAN_BIN = str(Path(LAKE).parent)

# 让 notebook / torchrun / subprocess 都能找到 lake/elan
os.environ["PATH"] = ELAN_BIN + ":" + os.environ.get("PATH", "")

print("ELAN_BIN =", ELAN_BIN)
print("lake exists =", Path(LAKE).exists())
print("elan exists =", Path(ELAN).exists())

# 快速自检（必须能跑通）
!cd {LEAN_PROJ} && which lake
!cd {LEAN_PROJ} && {LAKE} --version


print("WORK_DIR:", WORK_DIR)
print("TEST_RUN_DIR:", TEST_RUN_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("TOKENIZER_DIR:", TOKENIZER_DIR)
print("LEAN_PROJ:", LEAN_PROJ)


# ====== pass@k ======
K_ATTEMPTS = 32          # 每题生成 k 次完整尝试（pass@k）
GEN_SEED_BASE = 1234    # 生成随机种子基准（可复现）


PASSATK_TXT = f"{TEST_RUN_DIR}/passatk.txt"
print("K_ATTEMPTS:", K_ATTEMPTS)

print("PASSATK_TXT:", PASSATK_TXT)


# ====== 是否立刻插回 ======
IMMEDIATE_INSERT = 1   # 1: 逐洞生成后立即插回 sketch；0: 全洞生成完再一次性插（生成时互不可见）


# ====== 进度条设置 ======
TQDM_MININTERVAL = "120"       # 至少每 k 秒才刷新一次（默认 0.1s）
TQDM_MAXINTERVAL = "300"      # 最久 k 秒至少刷新一次（防止太久没动静）
TQDM_MINITERS = 1             # 至少处理 k 个迭代才允许刷新（对 it/s 很快时有效）
TQDM_SMOOTHING = "0.0"        # 可选：减少 ETA 抖动（不是降频，但更稳定）



# ====== 脚本保存目录 ======
SCRIPT_DIR = Path(WORK_DIR) / "Sequential_filling_script" / STAMP
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STEPWISE_GEN_PY = SCRIPT_DIR / "run_stepwise_generate.py"
MERGE_PY           = SCRIPT_DIR / "merge_jsonl_shards.py"
VERIFY_PY          = SCRIPT_DIR / "run_lean_verify.py"

ASSEMBLE_PY = SCRIPT_DIR / "assemble_holes.py"



In [ ]:
# Cell 2 脚本写入

from pathlib import Path

# ============ 1) run_stepwise_generate.py ============
gen_script = r'''
# run_stepwise_generate.py
import os, re, json, time, hashlib
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

EOFILL = "<eofill>"
HOLE = "<proof>"


# tqdm (progress bar) — safe import
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x



def count_holes(sketch: str) -> int:
    return sketch.count(HOLE)




def build_single_hole_sketch(orig: str, keep_idx: int, fills: dict, placeholder: str = "sorry") -> str:
    """
    orig: 原始多洞 sketch（只含 proof body，不含 theorem header）
    keep_idx: 要保留的洞（按 orig 中 <proof> 出现顺序，0-based）
    fills: 已经插入的洞 {hole_index: fill_code}；未插入的洞不在 dict 中
    placeholder: 其它未填洞用什么替换（默认 sorry）
    规则：已填洞保留 fill_code；未填洞中只有 keep_idx 保留 <proof>，其它换成 placeholder。
    """
    parts = orig.split(HOLE)
    n = len(parts) - 1
    if n <= 0:
        return orig
    if keep_idx < 0 or keep_idx >= n:
        keep_idx = n - 1

    out = []
    for i in range(n):
        out.append(parts[i])
        if i in fills:
            out.append(fills[i])
        else:
            out.append(HOLE if i == keep_idx else placeholder)
    out.append(parts[-1])
    return "".join(out)





SYSTEM_PROMPT = (
    "You are a Lean 4 proof completion model.\n"
    "Given a Lean 4 sketch that contains exactly ONE <proof> hole, you must output ONLY the Lean code "
    "that should replace the <proof>.\n"
    "Ignore any 'sorry' in the input; do not try to replace or modify it.\n"
    "Do not output explanations. Do not repeat the input.\n"
    "Do NOT include the token 'sorry' anywhere in your output.\n"
    "End your output with the token <eofill>.\n"
)

def resolve_adapter_dir(adapter_dir: str) -> str:
    """adapter_dir 可以是 checkpoint-xxx 或 out_dir 根目录（自动挑最新 checkpoint）"""
    p = Path(adapter_dir)
    if (p / "adapter_config.json").exists():
        return str(p)
    cands = []
    for ck in p.glob("checkpoint-*"):
        if (ck / "adapter_config.json").exists():
            m = re.search(r"checkpoint-(\d+)", ck.name)
            step = int(m.group(1)) if m else -1
            cands.append((step, ck))
    if cands:
        cands.sort()
        return str(cands[-1][1])
    raise FileNotFoundError(f"adapter_config.json not found under: {p}")

def load_jsonl(path: str):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            items.append(json.loads(ln))
    return items

def normalize_body_for_by(block: str) -> str:
    """保证 theorem ... := by 后面的 proof body 每行至少缩进 2 空格"""
    out = []
    for ln in block.splitlines():
        if ln.strip() == "":
            out.append("")
        elif ln.startswith("  "):
            out.append(ln)
        else:
            out.append("  " + ln)
    return "\n".join(out)

_CODEFENCE_RE = re.compile(r"^```.*?\n|\n```$", re.S)

def clean_fill(text: str) -> str:
    t = text.strip()
    t = _CODEFENCE_RE.sub("", t).strip()
    if EOFILL in t:
        t = t.split(EOFILL, 1)[0]
    t = t.replace(HOLE, "").strip("\n")
    return t.rstrip()

def _unbalanced(s: str) -> bool:
    return (s.count('[') > s.count(']')) or (s.count('(') > s.count(')')) or (s.count('{') > s.count('}'))

def _looks_truncated_lean(fill: str) -> bool:
    t = fill.rstrip()
    if t.endswith('[') or t.endswith(',') or t.endswith(':=') or t.endswith('by') or t.endswith('rw') or t.endswith('simp'):
        return True
    if 'rw [' in t and t.count('[') > t.count(']'):
        return True
    if 'linarith [' in t and t.count('[') > t.count(']'):
        return True
    return _unbalanced(t)

def find_first_hole_linecol(s: str):
    idx = s.find(HOLE)
    if idx < 0:
        return -1, -1
    line = s.count("\n", 0, idx)
    col = idx - (s.rfind("\n", 0, idx) + 1)
    return line, col



def build_prompt(tokenizer, header: str, theorem: str, sketch_state: str) -> str:
    user = (
        "Replace the <proof> in the following Lean 4 sketch.\n"
        "Ignore any occurrences of 'sorry'.\n\n"
        "Do NOT include the token 'sorry' anywhere in your output.\n"
        "[HEADER]\n"
        f"{header}\n\n"
        "[THEOREM+SKETCH]\n"
        f"{theorem}\n"
        f"{sketch_state}\n"
    )
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return f"<<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n<<USER>>\n{user}\n<</USER>>\n\n<<ASSISTANT>>\n"

def _stable_int_seed(s: str) -> int:
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16) % (2**31 - 1)

def _safe_key(s: str) -> str:
    return re.sub(r"[^0-9A-Za-z._-]+", "_", str(s))

def main():
    cfg_path = os.environ["RUN_CONFIG_PATH"]
    sketch_jsonl = os.environ["SKETCH_JSONL"]
    out_jsonl = os.environ["OUT_JSONL"]

    max_new = int(os.environ.get("MAX_NEW_TOKENS_PER_HOLE", "256"))
    max_holes = int(os.environ.get("MAX_HOLES_PER_PROOF", "32"))
    do_sample = bool(int(os.environ.get("DO_SAMPLE", "0")))
    temperature = float(os.environ.get("TEMPERATURE", "0.2"))
    top_p = float(os.environ.get("TOP_P", "0.95"))

    # pass@k：每题生成几次完整尝试
    k_attempts = int(os.environ.get("K_ATTEMPTS", "1"))
    seed_base  = int(os.environ.get("GEN_SEED_BASE", "1234"))

    adapter_dir = os.environ.get("ADAPTER_DIR", "")
    tokenizer_dir = os.environ.get("TOKENIZER_DIR", "")

    cfg = json.loads(Path(cfg_path).read_text(encoding="utf-8"))
    base_model = cfg["base_model"]
    out_dir_root = cfg.get("out_dir", "")

    if not adapter_dir:
        adapter_dir = out_dir_root
    adapter_dir = resolve_adapter_dir(adapter_dir)

    if not tokenizer_dir:
        tokenizer_dir = out_dir_root

    rank = int(os.environ.get("RANK", "0"))
    world = int(os.environ.get("WORLD_SIZE", "1"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)

    outp = Path(out_jsonl)
    # DDP 推荐传目录：out_jsonl=/path/to/gen_dir
    if outp.suffix == ".jsonl":
        out_file = outp
        out_file.parent.mkdir(parents=True, exist_ok=True)
    else:
        outp.mkdir(parents=True, exist_ok=True)
        out_file = outp / f"gen_rank{rank}.jsonl"

    if rank == 0:
        print("base_model:", base_model)
        print("tokenizer_dir:", tokenizer_dir)
        print("adapter_dir:", adapter_dir)
        print("out_file:", out_file)
        print("K_ATTEMPTS:", k_attempts)

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.add_special_tokens({"additional_special_tokens": [HOLE, EOFILL]})
    tokenizer.truncation_side = "left"
    tokenizer.padding_side = "left"

    eofill_id = tokenizer.convert_tokens_to_ids(EOFILL)
    stop_id = eofill_id if (eofill_id is not None and eofill_id >= 0) else tokenizer.eos_token_id

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_cfg,
        device_map={"": local_rank},
        torch_dtype=torch.bfloat16,
    )
    model.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    model.eval()

    data = load_jsonl(sketch_jsonl)

    # 若存在 split 字段则优先按 split==test；否则默认整文件就是测试集
    if any(("split" in x) for x in data):
        test = [x for x in data if x.get("split") == "test"]
        if not test:
            test = [x for x in data if x.get("split") == "valid"]
        if not test:
            test = data
    else:
        test = data



    t0 = time.time()
    n_lines = 0
    
    
    # =========  展开成“洞级任务”列表，然后按 rank 分片 =========
    immediate_insert = bool(int(os.environ.get("IMMEDIATE_INSERT", "0")))

    # =========  展开成“(题, attempt) 级任务”，rank 之间分题并行 =========
    tasks = []
    for ex in test:
        rid = ex.get("id", "")
        orig = ex.get("sketch", "") or ex.get("proof", "") or ""
        num_holes = orig.count(HOLE)
        if num_holes <= 0:
            continue
        num_holes = min(num_holes, max_holes)

        for attempt in range(k_attempts):
            tasks.append({
                "ex": ex,
                "rid": rid,
                "attempt": attempt,
                "num_holes": num_holes,
                "orig": orig,
            })

    total_groups_global = len(tasks)
    tasks = tasks[rank::world]
    total_groups_local = len(tasks)

    # 进度条按“洞”更直观：每个 group 有 num_holes 个洞
    total_holes_local = sum(t["num_holes"] for t in tasks)

    if rank == 0:
        print(f"[gen] groups_global={total_groups_global} world={world} immediate_insert={immediate_insert}")
    print(f"[gen] rank={rank} groups_local={total_groups_local} holes_local={total_holes_local}")


    t0 = time.time()
    n_lines = 0


    with open(out_file, "w", encoding="utf-8") as w:

        pbar = tqdm(
            total=total_holes_local,
            desc=f"gen rank{rank}",
            dynamic_ncols=True,
            disable=(rank != 0),   # Notebook 建议只显示 rank0，避免刷屏
            mininterval = float(os.environ.get("TQDM_MININTERVAL", "60")),      # ✅ 至少每 k 秒才刷新一次（默认 0.1s）
            maxinterval = float(os.environ.get("TQDM_MAXINTERVAL", "180")),     # ✅ 最久 k 秒至少刷新一次（防止太久没动静）
            miniters = int(os.environ.get("TQDM_MINITERS", "1")),               # 至少处理 k 个迭代才允许刷新（对 it/s 很快时有效）
            smoothing = float(os.environ.get("TQDM_SMOOTHING", "0.0")),         # ✅ 可选：减少 ETA 抖动（不是降频，但更稳定）
        )

        n_lines = 0

        for t in tasks:
            ex = t["ex"]
            rid = t["rid"]
            attempt = t["attempt"]
            num_holes = t["num_holes"]
            orig = t["orig"]

            header = ex.get("header", "") or ""
            theorem = ex.get("theorem", "") or ""

            # immediate_insert=1 时：fills_so_far 会逐步增长，后续洞可见前面填好的代码
            # immediate_insert=0 时：fills_so_far 固定为空，所有洞生成互不可见
            fills_so_far = {}

            for hi in range(num_holes):

                # 关键：构造“单洞可见”的输入 sketch
                if immediate_insert:
                    single = build_single_hole_sketch(orig, hi, fills_so_far, placeholder="sorry")
                else:
                    single = build_single_hole_sketch(orig, hi, {}, placeholder="sorry")

                seed = (seed_base + _stable_int_seed(f"{rid}__a{attempt}__h{hi}")) % (2**31 - 1)
                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

                prompt = build_prompt(tokenizer, header, theorem, single)
                enc = tokenizer(
                    prompt,
                    return_tensors="pt",
                    truncation=True,
                    max_length=int(cfg.get("max_len", 8192)),
                )
                input_ids = enc["input_ids"].to(model.device)
                attn = enc["attention_mask"].to(model.device)
                in_len = input_ids.shape[-1]

                do_sample_eff = do_sample or (k_attempts > 1)

                with torch.no_grad():
                    gen = model.generate(
                        input_ids=input_ids,
                        attention_mask=attn,
                        max_new_tokens=max_new,
                        do_sample=do_sample_eff,
                        temperature=temperature if do_sample_eff else None,
                        top_p=top_p if do_sample_eff else None,
                        eos_token_id=stop_id,
                        pad_token_id=tokenizer.pad_token_id,
                    )

                out_text = tokenizer.decode(gen[0][in_len:], skip_special_tokens=False)
                fill = clean_fill(out_text)

                # 截断/括号不平衡：重试一次（保留你原逻辑）
                if _looks_truncated_lean(fill):
                    gen2 = model.generate(
                        input_ids=input_ids,
                        attention_mask=attn,
                        max_new_tokens=max_new * 2,
                        do_sample=True,
                        temperature=max(0.7, temperature),
                        top_p=top_p,
                        eos_token_id=stop_id,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                    out_text2 = tokenizer.decode(gen2[0][in_len:], skip_special_tokens=False)
                    fill2 = clean_fill(out_text2)
                    if not _looks_truncated_lean(fill2):
                        out_text, fill = out_text2, fill2

                # ✅ 是否立即插回
                if immediate_insert:
                    fills_so_far[hi] = fill

                rec = dict(ex)
                rec.update({
                    "rid": rid,
                    "gen_attempt": attempt,
                    "hole_index": hi,
                    "num_holes": num_holes,
                    "gen_seed": seed,
                    "gen_rank": rank,
                    "gen_world": world,

                    "immediate_insert": bool(immediate_insert),

                    "orig_sketch": orig,
                    "single_sketch": single,   # 该洞生成时实际看到的 sketch（含/不含已插入代码）

                    "gen_fill": fill,
                    "gen_raw": out_text,
                })

                w.write(json.dumps(rec, ensure_ascii=False) + "\n")
                n_lines += 1
                pbar.update(1)

                if (n_lines % 1000) == 0:
                    pbar.set_postfix_str(f"written={n_lines}")

        pbar.close()

    if rank == 0:
        print(f"done. rank0 wrote {n_lines} HOLE-lines. elapsed={time.time()-t0:.1f}s")

if __name__ == "__main__":
    main()
'''

RUN_STEPWISE_GEN_PY.write_text(gen_script, encoding="utf-8")
print("✅ wrote:", RUN_STEPWISE_GEN_PY)


# ============ 2) merge_jsonl_shards.py ============
merge_script = r'''
import os, glob
from pathlib import Path

def main():
    in_dir = os.environ["IN_DIR"]
    out_jsonl = os.environ["OUT_JSONL"]

    files = sorted(glob.glob(str(Path(in_dir) / "gen_rank*.jsonl")))
    assert files, f"no shard files found in: {in_dir}"

    outp = Path(out_jsonl)
    outp.parent.mkdir(parents=True, exist_ok=True)

    cnt = 0
    with open(outp, "w", encoding="utf-8") as w:
        for fp in files:
            with open(fp, "r", encoding="utf-8") as f:
                for ln in f:
                    ln = ln.strip()
                    if not ln:
                        continue
                    w.write(ln + "\n")
                    cnt += 1
    print(f"merged {len(files)} shards -> {outp}  (lines={cnt})")

if __name__ == "__main__":
    main()

'''


MERGE_PY.write_text(merge_script, encoding="utf-8")
print("✅ wrote:", MERGE_PY)




# ============ 3) run_lean_verify.py ============


script = r'''
# run_lean_verify.py
import os, json, time, subprocess, re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


# tqdm (progress bar) — safe import
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x



def tail(s: str, n: int = 4000) -> str:
    if s is None:
        return ""
    return s[-n:]

def pick_code(ex: dict) -> str:
    # 优先取生成阶段的最终源码
    for k in ["gen_final_lean", "lean_code", "final_lean", "filled_lean", "lean_proof", "code"]:
        v = ex.get(k)
        if isinstance(v, str) and v.strip():
            return v

    # 兜底：尝试拼装
    header = ex.get("header", "") or ""
    theorem = ex.get("theorem", "") or ""
    proof = ex.get("proof_filled", "") or ex.get("proof", "") or ""
    return header + "\n\n" + theorem + "\n" + proof

def _safe_filename(s: str) -> str:
    return re.sub(r"[^0-9A-Za-z._-]+", "_", str(s))

def run_one(ex: dict, lean_proj: str, out_dir: Path, timeout: int, lake_bin: str):
    rid = ex.get("id", "")
    case_key = ex.get("gen_case_key") or str(rid)
    case_key = _safe_filename(case_key)

    code = pick_code(ex)
    out_path = out_dir / f"{case_key}.lean"
    out_path.write_text(code, encoding="utf-8")

    t0 = time.time()
    cmd = [lake_bin, "env", "lean", str(out_path)]

    # 输出：保留输入全部字段，再追加验证字段
    base = dict(ex)

    try:
        p = subprocess.run(
            cmd,
            cwd=lean_proj,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=os.environ.copy(),
        )
        ok = (p.returncode == 0)
        tag = "pass" if ok else "lean_error"

        base.update({
            "pass": bool(ok),
            "tag": tag,
            "returncode": p.returncode,
            "time_sec": time.time() - t0,
            "lean_path": str(out_path),
            "stdout_tail": tail(p.stdout),
            "stderr_tail": tail(p.stderr),
        })
        return base

    except subprocess.TimeoutExpired as e:
        base.update({
            "pass": False,
            "tag": "timeout",
            "returncode": None,
            "time_sec": time.time() - t0,
            "lean_path": str(out_path),
            "stdout_tail": tail(getattr(e, "stdout", "")),
            "stderr_tail": tail(getattr(e, "stderr", "")),
        })
        return base

    except FileNotFoundError as e:
        base.update({
            "pass": False,
            "tag": "lake_not_found",
            "returncode": None,
            "time_sec": time.time() - t0,
            "lean_path": str(out_path),
            "stdout_tail": "",
            "stderr_tail": str(e),
        })
        return base

    except Exception as e:
        base.update({
            "pass": False,
            "tag": "other",
            "returncode": None,
            "time_sec": time.time() - t0,
            "lean_path": str(out_path),
            "stdout_tail": "",
            "stderr_tail": repr(e),
        })
        return base

def main():
    gen_jsonl = os.environ["GEN_JSONL"]
    lean_proj = os.environ["LEAN_PROJ"]
    out_jsonl = os.environ["OUT_JSONL"]
    timeout = int(os.environ.get("TIMEOUT", "1200"))
    jobs = int(os.environ.get("JOBS", "8"))
    lake_bin = os.environ.get("LAKE", "lake")

    out_jsonl_p = Path(out_jsonl)
    out_jsonl_p.parent.mkdir(parents=True, exist_ok=True)

    out_dir = out_jsonl_p.parent / "lean_files"
    out_dir.mkdir(parents=True, exist_ok=True)

    items = []
    with open(gen_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            items.append(json.loads(line))

    t0 = time.time()
    passed = 0

    with open(out_jsonl, "w", encoding="utf-8") as w:
        with ThreadPoolExecutor(max_workers=jobs) as ex:
            futs = [ex.submit(run_one, it, lean_proj, out_dir, timeout, lake_bin) for it in items]
            
            
            pbar = tqdm(
                as_completed(futs),
                total=len(futs),
                desc="lean_verify",
                dynamic_ncols=True,
                mininterval = 10.0,   # ✅ 至少每 k 秒才刷新一次（默认 0.1s）
                maxinterval = 60.0,   # ✅ 最久 k 秒至少刷新一次（防止太久没动静）
                miniters = 50,        # ✅ 至少处理 k 个迭代才允许刷新（对 it/s 很快时有效）
                smoothing = 0.0,      # ✅ 可选：减少 ETA 抖动（不是降频，但更稳定）
            )
            
            done = 0
            for fut in pbar:
                r = fut.result()
                done += 1
                
                if r.get("pass"):
                    passed += 1
                w.write(json.dumps(r, ensure_ascii=False) + "\n")
                
                # 显示已通过数量
                if (done % 20) == 0:
                    pbar.set_postfix_str(f"done = {done}   pass = {passed}")
                    

    print(f"verified {len(items)} files. pass={passed} ({passed/len(items)*100:.2f}%). elapsed={time.time()-t0:.1f}s")
    print("out:", out_jsonl)

if __name__ == "__main__":
    main()

'''


VERIFY_PY.write_text(script, encoding="utf-8")
print("✅ Wrote:", VERIFY_PY)


assemble_script = r'''
# assemble_holes.py
import os, json, re
from collections import defaultdict
from pathlib import Path

HOLE = "<proof>"

def normalize_body_for_by(block: str) -> str:
    """保证 theorem ... := by 后面的 proof body 每行至少缩进 2 空格"""
    out = []
    for ln in block.splitlines():
        if ln.strip() == "":
            out.append("")
        elif ln.startswith("  "):
            out.append(ln)
        else:
            out.append("  " + ln)
    return "\n".join(out)

def safe_key(s: str) -> str:
    return re.sub(r"[^0-9A-Za-z._-]+", "_", str(s))

def load_jsonl(path: str):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            items.append(json.loads(ln))
    return items

def reassemble_by_index(orig: str, fills_by_index: dict, fill_upto: int, missing_placeholder: str = HOLE) -> str:
    """
    orig: 原始多洞 sketch（含多个 <proof>）
    fills_by_index: {hole_index: fill}
    fill_upto: 只填前 fill_upto 个洞（与生成时 num_holes / MAX_HOLES_PER_PROOF 对齐）
    missing_placeholder: 缺洞时填什么。默认填回 <proof>（保证结构不乱、且 verify 会 fail）
    """
    parts = orig.split(HOLE)
    n = len(parts) - 1
    if n <= 0:
        return orig

    # 只填前 fill_upto 个洞（通常 fill_upto == min(n, num_holes)）
    m = max(0, min(fill_upto, n))

    out = []
    for i in range(n):
        out.append(parts[i])
        if i < m:
            out.append(fills_by_index.get(i, missing_placeholder))
        else:
            out.append(HOLE)  # 超出生成范围的洞保持不动
    out.append(parts[-1])
    return "".join(out)

def main():
    """
    ENV:
      IN_JSONL  = holes_all.jsonl（洞级输出；每行一洞）
      OUT_JSONL = generated_all.jsonl（题级输出；每行一题/一次attempt）
      STRICT_COMPLETE (optional): "1" -> 洞不齐就跳过该组
    """
    
    skipped_jsonl = os.environ.get("SKIPPED_JSONL", "").strip()
    skipped_fp = None
    if skipped_jsonl:
        Path(skipped_jsonl).parent.mkdir(parents=True, exist_ok=True)
        skipped_fp = open(skipped_jsonl, "w", encoding="utf-8")

    
    in_jsonl = os.environ["IN_JSONL"]
    out_jsonl = os.environ["OUT_JSONL"]
    strict_complete = bool(int(os.environ.get("STRICT_COMPLETE", "0")))

    records = load_jsonl(in_jsonl)

    # group by (rid, attempt)
    groups = defaultdict(list)
    for ex in records:
        rid = ex.get("rid") or ex.get("id") or ex.get("name") or ""
        attempt = int(ex.get("gen_attempt", 0))
        groups[(rid, attempt)].append(ex)

    Path(out_jsonl).parent.mkdir(parents=True, exist_ok=True)

    assembled = 0
    skipped_incomplete = 0

    with open(out_jsonl, "w", encoding="utf-8") as w:
        for (rid, attempt), items in groups.items():
            # 以 hole_index 排序，保证确定性
            items.sort(key=lambda x: int(x.get("hole_index", 0)))

            base = dict(items[0])  # 继承原字段
            orig = base.get("orig_sketch") or base.get("sketch") or base.get("proof") or ""
            if not orig:
                continue

            # 期望洞数：优先 num_holes（与你生成时一致，能兼容 MAX_HOLES_PER_PROOF 截断）
            holes_expected = int(base.get("num_holes", orig.count(HOLE)))

            # 建立 hole_index -> record（若重复，保留第一个；也可改成挑更短/更长等策略）
            by_idx = {}
            duplicates = []
            for it in items:
                hi = int(it.get("hole_index", 0))
                if hi in by_idx:
                    duplicates.append(hi)
                    continue
                by_idx[hi] = it

            missing = [i for i in range(holes_expected) if i not in by_idx]
            holes_generated = len(by_idx)
            holes_complete = (len(missing) == 0) and (len(duplicates) == 0) and (holes_generated == holes_expected)

            if strict_complete and not holes_complete:
                skipped_incomplete += 1
                if skipped_fp:
                    skipped_fp.write(json.dumps({
                        "rid": rid,
                        "gen_attempt": attempt,
                        "holes_expected": holes_expected,
                        "holes_generated": holes_generated,
                        "missing_hole_indices": missing,
                        "duplicate_hole_indices": sorted(set(duplicates)),
                        "present_hole_indices": sorted(by_idx.keys()),
                        # 这两个有助于回溯原始数据
                        "id": base.get("id", None),
                        "name": base.get("name", None),
                    }, ensure_ascii=False) + "\n")
                continue


            # 回填用 fills_by_index
            fills_by_index = {hi: (by_idx[hi].get("gen_fill") or "") for hi in by_idx}

            # 组装回 orig（只填 holes_expected 个洞）
            cur = reassemble_by_index(orig, fills_by_index, fill_upto=holes_expected, missing_placeholder=HOLE)

            # 记录每个洞的 step（带各自 seed）
            gen_steps = []
            for hi in range(holes_expected):
                it = by_idx.get(hi)
                gen_steps.append({
                    "hole_index": hi,
                    "gen_fill": fills_by_index.get(hi, ""),
                    "gen_seed": (it.get("gen_seed") if it else None),
                    "gen_rank": (it.get("gen_rank") if it else None),
                    # 如果你觉得太大，可以删除下一行
                    "gen_raw":  (it.get("gen_raw")  if it else None),
                })

            # 拼接成完整 Lean
            header = base.get("header", "") or ""
            theorem = base.get("theorem", "") or ""
            final_body = normalize_body_for_by(cur)
            gen_final_lean = header.strip() + "\n\n" + theorem.rstrip() + "\n" + final_body + "\n"

            gen_uid = f"{rid}__a{attempt:02d}"
            base.update({
                "gen_uid": gen_uid,
                "gen_case_key": safe_key(gen_uid),

                "holes_expected": holes_expected,
                "holes_generated": holes_generated,
                "holes_complete": holes_complete,
                "missing_hole_indices": missing,
                "duplicate_hole_indices": sorted(set(duplicates)),

                "gen_num_fills": holes_expected,
                "gen_still_has_hole": (HOLE in cur),
                "gen_steps": gen_steps,
                "gen_final_lean": gen_final_lean,
            })

            w.write(json.dumps(base, ensure_ascii=False) + "\n")
            assembled += 1

    print(f"[assemble] groups_total={len(groups)} assembled={assembled} skipped_incomplete={skipped_incomplete}")
    print("[assemble] out ->", out_jsonl)
    
    if skipped_fp:
        skipped_fp.close()
        print("[assemble] skipped list ->", skipped_jsonl)


if __name__ == "__main__":
    main()
'''


ASSEMBLE_PY.write_text(assemble_script, encoding="utf-8")
print("✅ wrote:", ASSEMBLE_PY)


In [ ]:
# 3 —  GPU 分布式逐孔生成（torchrun）
cmd = (
    f'RUN_CONFIG_PATH="{RUN_CONFIG_PATH}" '
    f'SKETCH_JSONL="{SKETCH_JSONL}" '
    f'OUT_JSONL="{GEN_DIR}" '
    f'ADAPTER_DIR="{ADAPTER_DIR}" '
    f'TOKENIZER_DIR="{TOKENIZER_DIR}" '
    f'MAX_NEW_TOKENS_PER_HOLE="{MAX_NEW_TOKENS_PER_HOLE}" '
    f'MAX_HOLES_PER_PROOF="{MAX_HOLES_PER_PROOF}" '
    f'DO_SAMPLE="{DO_SAMPLE}" '
    f'TEMPERATURE="{TEMPERATURE}" '
    f'TOP_P="{TOP_P}" '
    f'K_ATTEMPTS="{K_ATTEMPTS}" '          
    f'GEN_SEED_BASE="{GEN_SEED_BASE}" '    
    f'IMMEDIATE_INSERT="{IMMEDIATE_INSERT}" '
    f'TQDM_MININTERVAL="{TQDM_MININTERVAL}" '
    f'TQDM_MAXINTERVAL="{TQDM_MAXINTERVAL}" '
    f'TQDM_MINITERS="{TQDM_MINITERS}" '
    f'TQDM_SMOOTHING="{TQDM_SMOOTHING}" '
    f'"{PYTHON}" -m torch.distributed.run --nproc_per_node=3 '
    f'"{RUN_STEPWISE_GEN_PY}"'
)
print(cmd)
!{cmd}


In [ ]:
# 4 — 合并 3 个 rank 的生成 shards
cmd = (
    f'IN_DIR="{GEN_DIR}" '
    f'OUT_JSONL="{MERGED_HOLES_JSONL}" '
    f'"{PYTHON}" "{MERGE_PY}"'
)
print(cmd)
!{cmd}


In [ ]:
# Cell 5
cmd = (
    f'STRICT_COMPLETE=1 '
    f'SKIPPED_JSONL="{SKIPPED_JSONL}" '
    f'IN_JSONL="{MERGED_HOLES_JSONL}" '
    f'OUT_JSONL="{MERGED_GEN_JSONL}" '
    f'"{PYTHON}" "{ASSEMBLE_PY}"'
)
print(cmd)
!{cmd}


In [ ]:
# Cell 6
cd {LEAN_PROJ} && {LAKE} build

In [ ]:
# Cell 7 — Lean 批量验证（输出 verify_results.jsonl）
cmd = (
f'GEN_JSONL="{MERGED_GEN_JSONL}" '
f'LEAN_PROJ="{LEAN_PROJ}" '
f'OUT_JSONL="{VERIFY_JSONL}" '
f'TIMEOUT="{VERIFY_TIMEOUT}" '
f'JOBS="{VERIFY_JOBS}" '
f'LAKE="{LAKE}" '
f'"{PYTHON}" "{VERIFY_PY}"'
)
print(cmd)
!{cmd}



In [ ]:
import json
import math
import statistics
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple, Union
from collections import Counter, defaultdict

# -----------------------------
# I/O: load records
# -----------------------------
def _load_records(inp: Union[str, Path, List[dict], Iterable[dict]]) -> List[dict]:
    if isinstance(inp, (list, tuple)):
        return [dict(x) for x in inp]
    if not isinstance(inp, (str, Path)):
        return [dict(x) for x in inp]

    s = str(inp)
    p = Path(s)
    if p.exists() and p.is_file():
        recs = []
        with open(p, "r", encoding="utf-8") as f:
            for ln in f:
                ln = ln.strip()
                if not ln:
                    continue
                recs.append(json.loads(ln))
        return recs

    recs = []
    for ln in s.splitlines():
        ln = ln.strip()
        if not ln:
            continue
        recs.append(json.loads(ln))
    return recs


# -----------------------------
# helpers: error extraction & categorization
# -----------------------------
_FIRST_ERROR_RE = re.compile(r"error:\s*(.+)", re.IGNORECASE)

def _get_error_text(rec: dict) -> str:
    return (rec.get("stdout_tail") or "") + "\n" + (rec.get("stderr_tail") or "")

def _extract_first_error_line(err_text: str) -> str:
    for ln in (err_text or "").splitlines():
        if "error:" in ln:
            return ln.strip()
    return ""

def _has_warning(rec: dict) -> bool:
    t = (rec.get("stdout_tail") or "") + "\n" + (rec.get("stderr_tail") or "")
    return ("warning:" in t)

def _classify_failure(tag: str, err_text: str, returncode: Any) -> str:
    tag = (tag or "").strip()
    t = (err_text or "").lower()

    known_fine = {
        "syntax_error", "unsolved_goals", "type_mismatch", "typeclass_failure",
        "timeout", "lake_not_found", "no_goals", "linarith_failed", "simp_failed",
        "rewrite_failed", "interval_cases", "tactic_failed", "other_rc1","unknown_identifier",
    }
    if tag in known_fine:
        return tag

    if tag in {"timeout", "lake_not_found"}:
        return tag

    if not t.strip() and returncode not in (0, None):
        return f"rc_{returncode}"

    if "unsolved goals" in t:
        return "unsolved_goals"
    
    if "unexpected token" in t or "expected ')'" in t or "expected ']'" in t or "expected term" in t:
        return "syntax_error"
    
    if "parser error" in t:
        return "syntax_error"
    
    if "type mismatch" in t:
        return "type_mismatch"
    
    # unknown identifier / constant（名字解析失败）
    if "unknown identifier" in t or "unknown constant" in t:
        return "unknown_identifier"

    
    if "failed to synthesize" in t or "failed to find instance" in t or "typeclass" in t:
        return "typeclass_failure"
    
    if "interval_cases failed" in t:
        return "interval_cases"
    
    if "linarith failed" in t:
        return "linarith_failed"
    
    if "tactic 'simp' failed" in t or "simp' failed" in t:
        return "simp_failed"
    
    if "tactic 'rewrite' failed" in t or "did not find instance of the pattern" in t:
        return "rewrite_failed"
    
    if "no goals to be solved" in t:
        return "no_goals"
    
    if "tactic failed" in t:
        return "tactic_failed"

    return tag if tag else "other"

def _percent(x: int, n: int) -> float:
    return 0.0 if n == 0 else (100.0 * x / n)

def _quantile(xs: List[float], q: float) -> Optional[float]:
    if not xs:
        return None
    xs = sorted(xs)
    if q <= 0:
        return xs[0]
    if q >= 1:
        return xs[-1]
    pos = (len(xs) - 1) * q
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    if lo == hi:
        return xs[lo]
    return xs[lo] * (hi - pos) + xs[hi] * (pos - lo)

def _time_pack(xs: List[float]) -> Dict[str, Optional[float]]:
    if not xs:
        return {"mean": None, "median": None, "p90": None, "p95": None, "max": None}
    return {
        "mean": sum(xs) / len(xs),
        "median": statistics.median(xs),
        "p90": _quantile(xs, 0.90),
        "p95": _quantile(xs, 0.95),
        "max": max(xs),
    }

# -----------------------------
# NEW helpers for pass@k setting
# -----------------------------
_GENUID_RE = re.compile(r"__a(\d+)$")

def _infer_case_id(rr: dict, idx: int) -> str:
    """
    题目级唯一 key（用于聚合多 attempt）。
    优先 id；否则 name/source_id/sample_index；再否则 idx（不推荐，但保底）。
    """
    rid = rr.get("id")
    if isinstance(rid, str) and rid.strip():
        return rid.strip()

    name = rr.get("name")
    source_id = rr.get("source_id")
    sample_index = rr.get("sample_index")
    parts = []
    if name not in (None, "", "null"):
        parts.append(f"name={name}")
    if source_id not in (None, "", "null"):
        parts.append(f"source_id={source_id}")
    if sample_index not in (None, "", "null"):
        parts.append(f"sample_index={sample_index}")

    if parts:
        return " | ".join(parts)

    return f"IDX={idx}"

def _infer_attempt_idx(rr: dict) -> int:
    """
    attempt index：
    1) gen_attempt
    2) 从 gen_uid / gen_case_key / lean_path 里解析 __aXX
    3) 默认 0
    """
    ga = rr.get("gen_attempt")
    if isinstance(ga, int):
        return ga
    if isinstance(ga, str) and ga.strip().isdigit():
        return int(ga.strip())

    for k in ["gen_uid", "gen_case_key", "lean_path"]:
        v = rr.get(k)
        if isinstance(v, str) and v:
            m = _GENUID_RE.search(v)
            if m:
                try:
                    return int(m.group(1))
                except Exception:
                    pass
    return 0

def _infer_gen_uid(case_id: str, attempt_idx: int) -> str:
    return f"{case_id}__a{attempt_idx:02d}"

def _sorted_attempts(attempt_recs: List[dict]) -> List[dict]:
    return sorted(attempt_recs, key=lambda x: (x["_attempt_idx"], str(x.get("gen_uid") or "")))



def _get_sketch_text(rr: dict, sketch_key_candidates: Tuple[str, ...] = ("sketch", "sketch_origin", "sketch_raw")) -> str:
    for k in sketch_key_candidates:
        v = rr.get(k)
        if isinstance(v, str) and v.strip():
            return v
    return ""

def _infer_hole_count_for_case(attempts: List[dict], proof_token: str = "<proof>") -> Optional[int]:
    """
    从一个 case 的 attempts 中推断“原 sketch 的洞数”。
    规则：找到第一条能拿到 sketch 的 attempt，用其 sketch.count("<proof>")。
    """
    for a in attempts:
        s = _get_sketch_text(a)
        if s:
            return s.count(proof_token)
    return None

def _hole_bucket_name(h: Optional[int]) -> str:
    if h is None:
        return "unknown_holes"
    if h == 1:
        return "single_hole"
    if h > 1:
        return "multi_hole"
    return "unknown_holes"



# -----------------------------
# main: analysis (multi-attempt aware)
# -----------------------------
def summarize_verify_records_multi(
    inp: Union[str, Path, List[dict], Iterable[dict]],
    bench_key_candidates: Tuple[str, ...] = ("source_bench", "source_bech"),
    top_k: int = 20,
    show_examples_per_reason: int = 3,
    pass_at_k_list: Tuple[int, ...] = (1, 2, 3, 5, 10),
    max_k_from_data: bool = True,
    return_summary: bool = True,
) -> Dict[str, Any]:
    """
    适配 verify 结果“同一题目多 attempt（pass@k）”的统计。

    输出两套视角：
      - Attempt-level: 每一行 verify 记录当一个样本
      - Case-level: 按题目聚合，计算 pass@k

    failure 也分：
      - attempt_failure_*: 对所有失败尝试做分类统计
      - case_failure_*: 对“在 max_k 下仍失败的题目”，用代表尝试（默认 attempt 最小）做分类统计
    """

    recs = _load_records(inp)
    total_lines = len(recs)

    # -------- normalize records (attempt-level) --------
    norm: List[dict] = []
    for idx, r in enumerate(recs):
        rr = dict(r)

        # bench
        bench = "UNKNOWN"
        for k in bench_key_candidates:
            if rr.get(k) not in (None, "", "null"):
                bench = rr.get(k)
                break

        # ids
        case_id = _infer_case_id(rr, idx)
        attempt_idx = _infer_attempt_idx(rr)

        # ensure gen_uid exists (do not overwrite if user already wrote it)
        if not (isinstance(rr.get("gen_uid"), str) and rr.get("gen_uid").strip()):
            rr["gen_uid"] = _infer_gen_uid(case_id, attempt_idx)

        passed = bool(rr.get("pass", False))
        tag = rr.get("tag", "") or ""
        rc = rr.get("returncode", None)
        time_sec = rr.get("time_sec", None)
        name = rr.get("name", "") or ""

        err_text = _get_error_text(rr)
        first_error = _extract_first_error_line(err_text)
        has_warn = _has_warning(rr)

        if passed:
            failure_cat = "pass"
        else:
            failure_cat = _classify_failure(tag, err_text, rc)

        rr["_bench"] = bench
        rr["_case_id"] = case_id
        rr["_attempt_idx"] = attempt_idx
        rr["_passed"] = passed
        rr["_tag"] = tag
        rr["_rc"] = rc
        rr["_time"] = time_sec if isinstance(time_sec, (int, float)) else None
        rr["_name"] = name
        rr["_failure_cat"] = failure_cat
        rr["_first_error"] = first_error
        rr["_has_warning"] = has_warn

        norm.append(rr)

    # group by bench -> case_id -> attempts
    bench_case_map: Dict[str, Dict[str, List[dict]]] = defaultdict(lambda: defaultdict(list))
    for rr in norm:
        bench_case_map[rr["_bench"]][rr["_case_id"]].append(rr)

    # overall case map
    overall_case_map: Dict[str, List[dict]] = defaultdict(list)
    for rr in norm:
        overall_case_map[rr["_case_id"]].append(rr)

    # determine max_k
    max_attempts_overall = 0
    for cid, attempts in overall_case_map.items():
        max_attempts_overall = max(max_attempts_overall, len(_sorted_attempts(attempts)))

    k_list = list(pass_at_k_list)
    if max_k_from_data and max_attempts_overall > 0:
        k_list.append(max_attempts_overall)
    k_list = sorted({k for k in k_list if isinstance(k, int) and k >= 1})
    if max_attempts_overall > 0:
        k_list = [k for k in k_list if k <= max_attempts_overall] or [max_attempts_overall]
    max_k = max(k_list) if k_list else 1

    # -------- summarizers --------
    def _summ_attempt_level(group_attempts: List[dict]) -> Dict[str, Any]:
        n = len(group_attempts)
        passed_n = sum(1 for x in group_attempts if x["_passed"])
        failed_n = n - passed_n
        warn_n = sum(1 for x in group_attempts if x["_has_warning"])

        fail_cats = [x["_failure_cat"] for x in group_attempts if not x["_passed"]]
        fail_counter = Counter(fail_cats)

        tag_counter = Counter([x["_tag"] or "" for x in group_attempts if not x["_passed"]])
        rc_counter = Counter([str(x["_rc"]) for x in group_attempts if not x["_passed"]])

        first_errs = [x["_first_error"] for x in group_attempts if (not x["_passed"] and x["_first_error"])]
        first_err_counter = Counter(first_errs)

        fail_names = [x["_name"] for x in group_attempts if (not x["_passed"] and x["_name"])]
        fail_name_counter = Counter(fail_names)

        times_all = [x["_time"] for x in group_attempts if isinstance(x["_time"], (int, float))]
        times_pass = [x["_time"] for x in group_attempts if x["_passed"] and isinstance(x["_time"], (int, float))]
        times_fail = [x["_time"] for x in group_attempts if (not x["_passed"]) and isinstance(x["_time"], (int, float))]

        fail_ratios = {k: _percent(v, failed_n) for k, v in fail_counter.most_common()}

        return {
            "attempt_n": n,
            "attempt_passed": passed_n,
            "attempt_failed": failed_n,
            "attempt_pass_rate": _percent(passed_n, n),
            "attempt_warning_count": warn_n,
            "attempt_warning_rate": _percent(warn_n, n),
            "attempt_failure_category_counts": dict(fail_counter),
            "attempt_failure_category_ratios_over_failed(%)": fail_ratios,
            "attempt_raw_tag_counts": dict(tag_counter),
            "attempt_returncode_counts": dict(rc_counter),
            "attempt_top_first_error": first_err_counter.most_common(top_k),
            "attempt_top_failed_names": fail_name_counter.most_common(top_k),
            "attempt_time_all": _time_pack(times_all),
            "attempt_time_pass": _time_pack(times_pass),
            "attempt_time_fail": _time_pack(times_fail),
        }

    def _summ_case_level(case_map: Dict[str, List[dict]]) -> Dict[str, Any]:
        """
        case_map: case_id -> list of attempts
        """
        case_n = len(case_map)
        
        
        # NEW: hole-bucket pass@k
        hole_bucket_counts = Counter()  # bucket -> number of cases
        pass_at_k_by_hole: Dict[str, Dict[int, int]] = defaultdict(lambda: defaultdict(int))  # bucket -> k -> solved_count

        

        # pass@k
        pass_at_k: Dict[int, Dict[str, Any]] = {}
        solved_cases_at_max_k: List[str] = []
        unsolved_cases_at_max_k: List[str] = []

        # for optional stats
        first_pass_attempt_pos: List[int] = []   # 1-based position in sorted attempts (for cases that solved)
        first_pass_time: List[float] = []        # time_sec of first passing attempt (if exists)

        # representative record for unsolved cases at max_k (used in failure breakdown)
        rep_unsolved_records: List[dict] = []

        for cid, attempts in case_map.items():
            ats = _sorted_attempts(attempts)
            
            
            # NEW: determine hole bucket for this case
            holes = _infer_hole_count_for_case(ats, proof_token="<proof>")
            hb = _hole_bucket_name(holes)
            hole_bucket_counts[hb] += 1


            # compute solved position/time
            solved_pos = None
            solved_time = None
            for j, a in enumerate(ats, start=1):
                if a["_passed"]:
                    solved_pos = j
                    solved_time = a["_time"] if isinstance(a["_time"], (int, float)) else None
                    break
            if solved_pos is not None:
                first_pass_attempt_pos.append(solved_pos)
                if solved_time is not None:
                    first_pass_time.append(solved_time)

            # pass@k decisions
            for k in k_list:
                topk = ats[:k]
                ok = any(a["_passed"] for a in topk)
                pass_at_k.setdefault(k, {"solved": 0})
                if ok:
                    pass_at_k[k]["solved"] += 1

                # NEW: hole-bucket pass@k
                if ok:
                    pass_at_k_by_hole[hb][k] += 1


            # max_k solved/unsolved sets
            maxk_ok = any(a["_passed"] for a in ats[:max_k])
            if maxk_ok:
                solved_cases_at_max_k.append(cid)
            else:
                unsolved_cases_at_max_k.append(cid)
                # representative attempt: smallest attempt_idx among first max_k
                rep = ats[0] if ats else None
                if rep is not None:
                    rep_unsolved_records.append(rep)

        # finalize pass@k rates
        pass_at_k_rates = {}
        for k in k_list:
            solved = pass_at_k.get(k, {}).get("solved", 0)
            pass_at_k_rates[k] = {
                "solved_cases": solved,
                "total_cases": case_n,
                "pass_at_k(%)": _percent(solved, case_n),
            }
            
            
            
        # NEW: finalize hole-bucket pass@k rates
        pass_at_k_by_hole_rates: Dict[str, Dict[int, Dict[str, Any]]] = {}
        for hb, total_hb in hole_bucket_counts.items():
            per_k = {}
            for k in k_list:
                solved = int(pass_at_k_by_hole[hb].get(k, 0))
                per_k[k] = {
                    "solved_cases": solved,
                    "total_cases": int(total_hb),
                    "pass_at_k(%)": _percent(solved, int(total_hb)),
                }
            pass_at_k_by_hole_rates[hb] = per_k


        # failure breakdown on unsolved cases at max_k
        fail_cats = [x["_failure_cat"] for x in rep_unsolved_records]  # all are failed
        fail_counter = Counter(fail_cats)
        failed_n = len(rep_unsolved_records)
        fail_ratios = {k: _percent(v, failed_n) for k, v in fail_counter.most_common()}

        # first_error top (unsolved case reps)
        first_errs = [x["_first_error"] for x in rep_unsolved_records if x["_first_error"]]
        first_err_counter = Counter(first_errs)

        # failed names top (unsolved case reps)
        fail_names = [x["_name"] for x in rep_unsolved_records if x["_name"]]
        fail_name_counter = Counter(fail_names)

        return {
            "case_n": case_n,
            "k_list": k_list,
            "max_k": max_k,
            "pass_at_k": pass_at_k_rates,
            
            "pass_at_k_by_hole": pass_at_k_by_hole_rates,
            "hole_bucket_case_counts": dict(hole_bucket_counts),

            "case_solved_at_max_k": len(solved_cases_at_max_k),
            "case_unsolved_at_max_k": len(unsolved_cases_at_max_k),
            "case_pass_at_max_k(%)": _percent(len(solved_cases_at_max_k), case_n),

            # how fast it solves (optional)
            "case_first_pass_attempt_pos_stats": {
                "mean": (sum(first_pass_attempt_pos) / len(first_pass_attempt_pos)) if first_pass_attempt_pos else None,
                "median": statistics.median(first_pass_attempt_pos) if first_pass_attempt_pos else None,
                "p90": _quantile([float(x) for x in first_pass_attempt_pos], 0.90) if first_pass_attempt_pos else None,
                "max": max(first_pass_attempt_pos) if first_pass_attempt_pos else None,
                "n_solved": len(first_pass_attempt_pos),
            },
            "case_first_pass_time_stats": _time_pack(first_pass_time),

            # unsolved reason breakdown (representative attempt per case)
            "case_failure_category_counts_unsolved_at_max_k": dict(fail_counter),
            "case_failure_category_ratios_over_unsolved(%)": fail_ratios,
            "case_top_first_error_unsolved_at_max_k": first_err_counter.most_common(top_k),
            "case_top_failed_names_unsolved_at_max_k": fail_name_counter.most_common(top_k),

            # store reps for examples
            "_rep_unsolved_records": rep_unsolved_records,
        }

    def _fmt_time(d: Dict[str, Optional[float]]) -> str:
        def f(x):
            return "NA" if x is None else f"{x:.3f}"
        return f"mean={f(d['mean'])}  median={f(d['median'])}  p90={f(d['p90'])}  p95={f(d['p95'])}  max={f(d['max'])}"

    def _print_report(title: str, attempts: List[dict], case_map: Dict[str, List[dict]]):
        attempt_s = _summ_attempt_level(attempts)
        case_s = _summ_case_level(case_map)

        print("\n" + "=" * 90)
        print(f"[{title}]")
        print(f"Attempts: Total={attempt_s['attempt_n']} | PASS={attempt_s['attempt_passed']} ({attempt_s['attempt_pass_rate']:.2f}%) | FAIL={attempt_s['attempt_failed']} ({100.0 - attempt_s['attempt_pass_rate']:.2f}%)")
        print(f"Cases   : Total={case_s['case_n']} | Solved@max_k={case_s['case_solved_at_max_k']} ({case_s['case_pass_at_max_k(%)']:.2f}%) | Unsolved@max_k={case_s['case_unsolved_at_max_k']}")
        print(f"Warnings(attempt-level)={attempt_s['attempt_warning_count']} ({attempt_s['attempt_warning_rate']:.2f}%)")
        print("-" * 90)

        # pass@k table
        print("pass@k (case-level):")
        for k in case_s["k_list"]:
            r = case_s["pass_at_k"][k]
            print(f"  - pass@{k:<2d}: {r['solved_cases']:5d}/{r['total_cases']:<5d} ({r['pass_at_k(%)']:.2f}%)")

            
            
        # NEW: pass@k by hole bucket
        if "pass_at_k_by_hole" in case_s:
            print("\npass@k by hole-bucket (case-level):")
            counts = case_s.get("hole_bucket_case_counts", {})
            for hb in ["single_hole", "multi_hole", "unknown_holes"]:
                if hb not in case_s["pass_at_k_by_hole"]:
                    continue
                total_hb = counts.get(hb, 0)
                print(f"  [{hb}] total_cases={total_hb}")
                for k in case_s["k_list"]:
                    r = case_s["pass_at_k_by_hole"][hb][k]
                    print(f"    - pass@{k:<2d}: {r['solved_cases']:5d}/{r['total_cases']:<5d} ({r['pass_at_k(%)']:.2f}%)")

            
            
        # attempt-level failure categories
        if attempt_s["attempt_failed"] > 0:
            print("\nAttempt-level failure categories (count / % over failed attempts):")
            for k, c in sorted(attempt_s["attempt_failure_category_counts"].items(), key=lambda x: x[1], reverse=True):
                pct = attempt_s["attempt_failure_category_ratios_over_failed(%)"].get(k, 0.0)
                print(f"  - {k:18s} : {c:5d}  ({pct:6.2f}%)")

        # case-level unsolved failure categories at max_k
        if case_s["case_unsolved_at_max_k"] > 0:
            print(f"\nCase-level failure categories (UNSOLVED at max_k={case_s['max_k']}, representative attempt) (count / % over unsolved cases):")
            for k, c in sorted(case_s["case_failure_category_counts_unsolved_at_max_k"].items(), key=lambda x: x[1], reverse=True):
                pct = case_s["case_failure_category_ratios_over_unsolved(%)"].get(k, 0.0)
                print(f"  - {k:18s} : {c:5d}  ({pct:6.2f}%)")

            # examples per reason (unsolved reps)
            if show_examples_per_reason > 0:
                print("\nExamples per failure category (UNSOLVED reps):")
                by_cat = defaultdict(list)
                for x in case_s["_rep_unsolved_records"]:
                    by_cat[x["_failure_cat"]].append(x)

                for cat, _ in sorted(case_s["case_failure_category_counts_unsolved_at_max_k"].items(), key=lambda x: x[1], reverse=True):
                    xs = by_cat.get(cat, [])
                    if not xs:
                        continue
                    print(f"  * {cat}:")
                    for ex in xs[:show_examples_per_reason]:
                        print(f"      id={ex.get('id')}  gen_uid={ex.get('gen_uid')}  attempt={ex.get('gen_attempt')}  name={ex.get('name')}  rc={ex.get('returncode')}")
                        fe = ex.get("_first_error") or ""
                        if fe:
                            print(f"        first_error: {fe[:200]}")
                        else:
                            et = _get_error_text(ex).strip().splitlines()
                            if et:
                                print(f"        err_line: {et[0][:200]}")

        # time stats
        print("\nTime stats (sec) (attempt-level):")
        print("  all : " + _fmt_time(attempt_s["attempt_time_all"]))
        print("  pass: " + _fmt_time(attempt_s["attempt_time_pass"]))
        print("  fail: " + _fmt_time(attempt_s["attempt_time_fail"]))

        print("\nFirst-pass attempt position stats (case-level, solved cases only):")
        ap = case_s["case_first_pass_attempt_pos_stats"]
        if ap["n_solved"] > 0:
            print(f"  n_solved={ap['n_solved']} | mean={ap['mean']:.3f} | median={ap['median']:.3f} | p90={ap['p90']:.3f} | max={ap['max']}")
        else:
            print("  n_solved=0")

        # top first_error (unsolved reps)
        if case_s["case_top_first_error_unsolved_at_max_k"]:
            print(f"\nTop first_error (UNSOLVED reps, max_k={case_s['max_k']}):")
            for msg, c in case_s["case_top_first_error_unsolved_at_max_k"][:min(top_k, 10)]:
                print(f"  - ({c}) {msg[:200]}")

        # top failed names (unsolved reps)
        if case_s["case_top_failed_names_unsolved_at_max_k"]:
            print(f"\nTop failed theorem names (UNSOLVED reps, max_k={case_s['max_k']}):")
            for nm, c in case_s["case_top_failed_names_unsolved_at_max_k"][:min(top_k, 10)]:
                print(f"  - ({c}) {nm}")

    # -------- overall report --------
    overall_attempts = norm
    overall_case_map2 = overall_case_map

    overall_attempt_s = _summ_attempt_level(overall_attempts)
    overall_case_s = _summ_case_level(overall_case_map2)
    _print_report("OVERALL", overall_attempts, overall_case_map2)

    # -------- by bench report --------
    by_bench = {}
    for bench, case_map in sorted(bench_case_map.items(), key=lambda kv: len(kv[1]), reverse=True):
        attempts = []
        for cid, ats in case_map.items():
            attempts.extend(ats)

        attempt_s = _summ_attempt_level(attempts)
        case_s = _summ_case_level(case_map)
        _print_report(f"BENCH={bench}", attempts, case_map)

        # 去掉内部字段再放 summary
        case_s2 = dict(case_s)
        case_s2.pop("_rep_unsolved_records", None)

        by_bench[bench] = {
            "attempt_level": attempt_s,
            "case_level": case_s2,
        }

    overall_case_s2 = dict(overall_case_s)
    overall_case_s2.pop("_rep_unsolved_records", None)

    summary = {
        "meta": {
            "total_lines(attempts)": total_lines,
            "total_cases": len(overall_case_map2),
            "k_list": k_list,
            "max_k": max_k,
        },
        "overall": {
            "attempt_level": overall_attempt_s,
            "case_level": overall_case_s2,
        },
        "by_bench": by_bench,
    }
    return summary if return_summary else {}


In [ ]:
VERIFY_JSONL = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2/test_runs/Sequential_filling_test/Sequential_filling_test_results/verify_results.jsonl"

In [ ]:
summary = summarize_verify_records_multi(
    VERIFY_JSONL,
    pass_at_k_list=(1, 12, 24, 32),
    show_examples_per_reason=3,
)
